### Data Pre-Processing

Load and preprocessed data, then save it to `processed_data` folder. Data in this folder should be used by subsequent assignment questions.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import warnings

warnings.filterwarnings("ignore")

# Load stock prices, market index prices, and the 10-year risk-free rate
prices = pd.read_csv("./original_data/prices.csv", index_col="Date", parse_dates=True)
market = pd.read_csv("./original_data/market.csv", index_col="Date", parse_dates=True)
rf_raw = pd.read_csv(
    "./original_data/DGS10.csv", index_col="observation_date", parse_dates=True
)

# Convert the 10-year yield (annual %) to a daily log risk-free rate
rf_raw.columns = ["yield_pct"]
rf_raw["rf_daily"] = (
    np.log(1 + pd.to_numeric(rf_raw["yield_pct"], errors="coerce") / 100) / 252
)
rf_raw = rf_raw.dropna(subset=["rf_daily"])

# Filter all series to the most recent 10 years
end_date = prices.index.max()
start_date = end_date - pd.DateOffset(years=10)
prices = prices.loc[start_date:end_date]
market = market.loc[start_date:end_date]
rf_raw = rf_raw.loc[start_date:end_date]

# Drop columns with more than 20% missing values, then forward-fill and drop any remaining gaps
prices = prices.dropna(thresh=int(0.80 * len(prices)), axis=1).ffill().dropna(axis=1)

# Align all three series to their common trading dates
common = prices.index.intersection(market.index).intersection(rf_raw.index)
prices = prices.loc[common]
mkt_px = market.loc[common].iloc[:, 0]
rf_d = rf_raw.loc[common, "rf_daily"]

# Compute daily log returns for stocks and the market: r_t = ln(P_t / P_{t-1})
log_ret = np.log(prices / prices.shift(1)).dropna()
mkt_ret = np.log(mkt_px / mkt_px.shift(1)).dropna()

# Re-align after differencing (first row becomes NaN)
common2 = log_ret.index.intersection(mkt_ret.index).intersection(rf_d.index)
log_ret = log_ret.loc[common2]
mkt_ret = mkt_ret.loc[common2]
rf_d = rf_d.loc[common2]

# Save daily log returns
log_ret.to_csv("./processed_data/stock_log_ret_daily.csv")
mkt_ret.to_csv("./processed_data/mkt_log_ret_daily.csv", header=["mkt"])

# Save daily risk-free rate
rf_d.to_csv("./processed_data/rf_daily.csv", header=["rf_daily"])

# Compute **daily excess log returns** (return minus daily risk-free rate)
stock_excess_daily = log_ret.subtract(rf_d, axis=0)
mkt_excess_daily = mkt_ret - rf_d

# Aggregate daily log returns to monthly by summing within each calendar month
log_ret_m = log_ret.resample("M").sum()
mkt_ret_m = mkt_ret.resample("M").sum()
rf_m = rf_d.resample("M").sum()

# Compute monthly excess returns (return minus risk-free rate)
mkt_excess = (mkt_ret_m - rf_m).dropna()
stock_excess = log_ret_m.subtract(rf_m, axis=0)
stock_excess = stock_excess.loc[mkt_excess.index].dropna(axis=1)

# Save MONTHLY excess returns
stock_excess.to_csv("./processed_data/stock_excess_monthly.csv")
mkt_excess.to_csv("./processed_data/mkt_excess_monthly.csv", header=["mkt_excess"])
rf_m.to_csv("./processed_data/rf_monthly.csv", header=["rf"])

# Save DAILY excess returns
stock_excess_daily.to_csv("./processed_data/stock_excess_daily.csv")
mkt_excess_daily.to_csv(
    "./processed_data/mkt_excess_daily.csv", header=["mkt_excess_daily"]
)

print(
    "Shared data saved: stock_excess_monthly.csv, mkt_excess_monthly.csv, rf_monthly.csv, "
    "stock_excess_daily.csv, mkt_excess_daily.csv"
)


Shared data saved: stock_excess_monthly.csv, mkt_excess_monthly.csv, rf_monthly.csv, stock_excess_daily.csv, mkt_excess_daily.csv
